# Generative LLMs vs. Released Base Decision Models on `Vaccines` (OLS AMCE Baseline)

**Pure CPU analysis notebook** (zero GPU or model loading). Reads saved KEV predictions from `llm-baseline/preds/` (generated by `predict_kev.py`) and saved Gemini responses from `code/proprietary-llms/gemini/data_sample1000/`.

- **Ground truth:** Full-sample human population OLS AMCEs on Kreps et al. (2020) (`N = 1,971` respondents, `9,855` paired choice sets, $d = 11$ treatment indicators) using the exact `beta_ols.covid` operator used throughout the main paper.
- **Evaluation subsample:** The exact 200 respondents (`n = 1,000` choice sets, `seed = 42`, matching Wang 2026).
- **Models compared (all zero-shot / base):**
  1. **Human subsample** (200 respondents, finite-sample reference).
  2. **Generative LLMs** (`Gemini 3.5 Flash-Lite` minimal/high thinking; `Gemini 4 Argon` low/high thinking; both random and 9-attribute respondent personas).
  3. **Released base decision models** (`KEV-0.8B`, `KEV-4B`, `KEV-9B`) at their **shipped calibrated temperature $T_{\mathrm{cal}}$**, plus argmax one-hot and 1-draw Monte Carlo ablations (and uncalibrated $T = 1.0$ for reference).

In [42]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

REPO_ROOT = Path("/home/dw/github/decision-models-finetuning")
CODE_DIR = REPO_ROOT / "code"
PAPER_DIR = CODE_DIR / "papers" / "sim-respondents"
SRC_DIR = PAPER_DIR / "src"
OUT_DIR = PAPER_DIR / "analysis" / "llm-baseline"
PREDS_DIR = OUT_DIR / "preds"
GEMINI_DIR = CODE_DIR / "proprietary-llms" / "gemini" / "data_sample1000"

for p in (str(CODE_DIR), str(SRC_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

import beta_ols.covid as ds_covid
from beta_ols.evaluation import build_population_eval, flat_probs_from_predictions

GEMINI_RUNS = [
    ("Gemini 3.5 Flash-Lite", "Minimal thinking", "Random",     "gemini-3.5-flash-lite_MINIMAL_random.jsonl"),
    ("Gemini 3.5 Flash-Lite", "Minimal thinking", "Respondent", "gemini-3.5-flash-lite_MINIMAL_persona.jsonl"),
    ("Gemini 3.5 Flash-Lite", "High thinking",    "Random",     "gemini-3.5-flash-lite_HIGH_random.jsonl"),
    ("Gemini 3.5 Flash-Lite", "High thinking",    "Respondent", "gemini-3.5-flash-lite_HIGH_persona.jsonl"),
    ("Gemini 4 Argon",        "Low thinking",     "Random",     "gemini-4-argon_LOW_random.jsonl"),
    ("Gemini 4 Argon",        "Low thinking",     "Respondent", "gemini-4-argon_LOW_persona.jsonl"),
    ("Gemini 4 Argon",        "High thinking",    "Random",     "gemini-4-argon_HIGH_random.jsonl"),
    ("Gemini 4 Argon",        "High thinking",    "Respondent", "gemini-4-argon_HIGH_persona.jsonl"),
]

meta_path = PREDS_DIR / "meta.json"
if not meta_path.exists():
    raise FileNotFoundError(f"Missing {meta_path}. Run predict_kev.py on GPU first.")
kev_meta = json.loads(meta_path.read_text())
print("Loaded KEV prediction metadata:")
for slug, info in kev_meta["models"].items():
    print(f"  {info['display_name']:9s} | T_cal={info['calibrated_temperature']:.4f} | file={info['file']}")

Loaded KEV prediction metadata:
  KEV-0.8B  | T_cal=2.3511 | file=kev-0.8b_covid_sample200.parquet
  KEV-4B    | T_cal=2.4061 | file=kev-4b_covid_sample200.parquet
  KEV-9B    | T_cal=2.1936 | file=kev-9b_covid_sample200.parquet


## 1. Load Full Human Ground Truth & Verify Exact 200-Respondent Subsample Alignment

In [43]:
full_choice_df = ds_covid.load_choice_df().copy()
full_choice_df["respondent"] = full_choice_df["respondent"].astype(int)
full_choice_df["pair_id"] = full_choice_df.groupby("respondent").cumcount() + 1

if len(full_choice_df) != 9855 or full_choice_df["respondent"].nunique() != 1971:
    raise ValueError(f"Unexpected covid choice_df shape: {len(full_choice_df)} rows")

full_reg_rows = ds_covid.regression_rows(full_choice_df)
full_specs = ds_covid.regression_specs(full_reg_rows)
full_records = [{"CASEID": str(r.CASEID), "qid": str(r.qid)} for r in full_choice_df.itertuples(index=False)]
full_eval_ops = build_population_eval(ds_covid, full_choice_df, full_specs, full_records)

term_labels = list(full_specs[ds_covid.NAME].target_labels)
human_full_betas = full_eval_ops["human_betas"][ds_covid.NAME].numpy().astype(np.float64)

all_resps = np.sort(full_choice_df["respondent"].unique())
sampled_resps = np.sort(np.random.default_rng(42).choice(all_resps, size=200, replace=False))
sampled_set = {int(r) for r in sampled_resps}

for _, _, _, fname in GEMINI_RUNS:
    fpath = GEMINI_DIR / fname
    if not fpath.exists():
        raise FileNotFoundError(f"Missing Gemini file: {fpath}")
    rows = [json.loads(line) for line in fpath.read_text().splitlines()]
    if {int(r["respondent"]) for r in rows} != sampled_set or len(rows) != 1000:
        raise ValueError(f"{fname}: respondent IDs or row count mismatch")

sub_choice_df = full_choice_df[full_choice_df["respondent"].isin(sampled_set)].reset_index(drop=True)
if len(sub_choice_df) != 1000 or sub_choice_df["respondent"].nunique() != 200:
    raise ValueError(f"Expected 1,000 rows across 200 respondents, got {len(sub_choice_df)}")

print(f"Full human ground truth: {len(full_choice_df):,} tasks, {len(term_labels)} AMCEs")
print(f"Verified 200-respondent subsample ({len(sub_choice_df):,} tasks) matches all {len(GEMINI_RUNS)} Gemini files.")

Full human ground truth: 9,855 tasks, 11 AMCEs
Verified 200-respondent subsample (1,000 tasks) matches all 8 Gemini files.


## 2. Helper: Score Any Task-Level Probability Array with Exact `beta_ols.covid` OLS Operator

In [44]:
def score_predictions_ols(
    df_tasks: pd.DataFrame,
    probs_list: list[np.ndarray],
    human_gt_betas: np.ndarray,
) -> tuple[dict[str, float], np.ndarray]:
    if len(df_tasks) != len(probs_list):
        raise ValueError(f"Length mismatch: {len(df_tasks)} tasks vs {len(probs_list)} predictions")
    reg_rows = ds_covid.regression_rows(df_tasks)
    specs = ds_covid.regression_specs(reg_rows)
    records = [{"CASEID": str(r.CASEID), "qid": str(r.qid)} for r in df_tasks.itertuples(index=False)]
    eval_ops = build_population_eval(ds_covid, df_tasks, specs, records)
    probs_flat = flat_probs_from_predictions(eval_ops, probs_list)
    model_betas = (eval_ops["operators"][0]["B"] @ probs_flat).numpy().astype(np.float64)

    diff = model_betas - human_gt_betas
    abs_diff = np.abs(diff)
    if np.any(human_gt_betas == 0.0):
        raise ValueError("Zero human ground-truth beta encountered")

    metrics = {
        "n_tasks": int(len(df_tasks)),
        "opt_out_pct": float(np.mean([float(p[2]) for p in probs_list]) * 100.0),
        "beta_mse_x1e3": float(np.mean(diff ** 2) * 1000.0),
        "beta_mae": float(np.mean(abs_diff)),
        "beta_mape_pct": float(np.mean(abs_diff / np.abs(human_gt_betas)) * 100.0),
        "spearman_rho": float(spearmanr(model_betas, human_gt_betas).statistic),
        "sign_acc_pct": float(np.mean(np.sign(model_betas) == np.sign(human_gt_betas)) * 100.0),
    }
    return metrics, model_betas


summary_rows = []
betas_by_model: dict[str, np.ndarray] = {"Human (full, N=1,971)": human_full_betas}

# 200-respondent human finite-sample benchmark
human_sub_probs = [np.asarray(v, dtype=np.float64) for v in sub_choice_df["one_hot"]]
m_hum, b_hum = score_predictions_ols(sub_choice_df, human_sub_probs, human_full_betas)
summary_rows.append({
    "family": "Human benchmark",
    "model": "Survey subsample (200 resp.)",
    "config": "Observed choices",
    "persona": "Respondent",
    "readout": "Observed one-hot",
    "refusals": 0,
    **m_hum,
})
betas_by_model["Human (200 resp.)"] = b_hum
print(f"Human 200-resp subsample: beta_mse*1e3 = {m_hum['beta_mse_x1e3']:.3f}, opt_out = {m_hum['opt_out_pct']:.1f}%")

Human 200-resp subsample: beta_mse*1e3 = 0.455, opt_out = 20.3%


## 3. Score All Saved Gemini and Qwen Runs (`Gemini 3.5 Flash-Lite` & `Gemini 4 Argon`)

In [45]:
LETTER_TO_ONEHOT = {
    "A": np.array([1.0, 0.0, 0.0], dtype=np.float64),
    "B": np.array([0.0, 1.0, 0.0], dtype=np.float64),
    "C": np.array([0.0, 0.0, 1.0], dtype=np.float64),
}
MIN_VALID_FOR_OLS = 100


def score_llm_jsonl(
    jsonl_path: Path,
    *,
    family: str,
    model_name: str,
    config_name: str,
    persona_kind: str,
) -> None:
    """Exact shared parser for both Gemini and Qwen JSONL outputs: valid A/B/C enters OLS; invalid/refusals are recorded."""
    rows = [json.loads(line) for line in jsonl_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    by_key = {}
    for r in rows:
        key = (int(r["respondent"]), int(r["pair_id"]))
        if key in by_key:
            raise ValueError(f"{jsonl_path.name}: duplicate key {key}")
        by_key[key] = r["raw_response"].strip()

    valid_mask = []
    valid_probs = []
    invalid_or_refusals = 0
    for r in sub_choice_df.itertuples(index=False):
        key = (int(r.respondent), int(r.pair_id))
        if key not in by_key:
            raise KeyError(f"{jsonl_path.name}: missing key {key}")
        ans = by_key[key]
        if ans in LETTER_TO_ONEHOT:
            valid_mask.append(True)
            valid_probs.append(LETTER_TO_ONEHOT[ans])
        else:
            valid_mask.append(False)
            invalid_or_refusals += 1

    label = f"{model_name} ({config_name}, {persona_kind})"
    if len(valid_probs) < MIN_VALID_FOR_OLS:
        summary_rows.append({
            "family": family,
            "model": model_name,
            "config": config_name,
            "persona": persona_kind,
            "readout": "Discrete text",
            "refusals": invalid_or_refusals,
            "n_tasks": len(valid_probs),
            "opt_out_pct": float(np.mean([p[2] for p in valid_probs]) * 100.0) if valid_probs else float("nan"),
            "beta_mse_x1e3": float("nan"),
            "beta_mae": float("nan"),
            "beta_mape_pct": float("nan"),
            "spearman_rho": float("nan"),
            "sign_acc_pct": float("nan"),
        })
        print(
            f"{label:56s} | n={len(valid_probs):4d} (inv={invalid_or_refusals:3d}) "
            f"| SKIPPED OLS (<{MIN_VALID_FOR_OLS} valid responses)"
        )
        return

    df_valid = sub_choice_df.loc[valid_mask].reset_index(drop=True)
    metrics, betas = score_predictions_ols(df_valid, valid_probs, human_full_betas)
    summary_rows.append({
        "family": family,
        "model": model_name,
        "config": config_name,
        "persona": persona_kind,
        "readout": "Discrete text",
        "refusals": invalid_or_refusals,
        **metrics,
    })
    betas_by_model[label] = betas
    print(
        f"{label:56s} | n={metrics['n_tasks']:4d} (inv={invalid_or_refusals:3d}) "
        f"| opt_out={metrics['opt_out_pct']:5.1f}% | MSE*1e3={metrics['beta_mse_x1e3']:7.2f} | rho={metrics['spearman_rho']:.3f}"
    )


print("=== Gemini runs (data_sample1000) ===")
for model_name, config_name, persona_kind, fname in GEMINI_RUNS:
    score_llm_jsonl(
        GEMINI_DIR / fname,
        family="Generative LLM (Gemini)",
        model_name=model_name,
        config_name=config_name,
        persona_kind=persona_kind,
    )

qwen_files = sorted(PREDS_DIR.glob("qwen*.jsonl"))
if qwen_files:
    print("\n=== Qwen3.5 runs (preds/qwen*.jsonl) ===")
    for qpath in qwen_files:
        lines = [ln for ln in qpath.read_text(encoding="utf-8").splitlines() if ln.strip()]
        if len(lines) != len(sub_choice_df):
            print(f"  Skipping incomplete file {qpath.name} ({len(lines)}/{len(sub_choice_df)} rows)")
            continue
        first = json.loads(lines[0])
        model_name = first["model"].split("/")[-1]
        think_label = "Thinking on" if first["thinking_level"] == "THINKING_ON" else "Thinking off"
        stem_parts = qpath.stem.split("_")
        persona_raw = stem_parts[-1]
        if persona_raw not in ("respondent", "random"):
            raise ValueError(f"Cannot infer persona from filename: {qpath.name}")
        persona_label = "Respondent" if persona_raw == "respondent" else "Random"
        score_llm_jsonl(
            qpath,
            family="Generative LLM (Qwen)",
            model_name=model_name,
            config_name=think_label,
            persona_kind=persona_label,
        )
else:
    print("\n(No preds/qwen*.jsonl files found yet; run predict_qwen.py to add Qwen3.5 rows.)")

=== Gemini runs (data_sample1000) ===
Gemini 3.5 Flash-Lite (Minimal thinking, Random)         | n=1000 (inv=  0) | opt_out=  3.0% | MSE*1e3=  18.06 | rho=0.927
Gemini 3.5 Flash-Lite (Minimal thinking, Respondent)     | n=1000 (inv=  0) | opt_out= 21.9% | MSE*1e3=  10.64 | rho=0.936
Gemini 3.5 Flash-Lite (High thinking, Random)            | n=1000 (inv=  0) | opt_out=  0.0% | MSE*1e3=  19.05 | rho=0.891
Gemini 3.5 Flash-Lite (High thinking, Respondent)        | n=1000 (inv=  0) | opt_out=  4.8% | MSE*1e3=  12.28 | rho=0.900
Gemini 4 Argon (Low thinking, Random)                    | n=1000 (inv=  0) | opt_out=  0.3% | MSE*1e3=  20.55 | rho=0.955
Gemini 4 Argon (Low thinking, Respondent)                | n=1000 (inv=  0) | opt_out= 12.2% | MSE*1e3=  16.50 | rho=0.927
Gemini 4 Argon (High thinking, Random)                   | n= 984 (inv= 16) | opt_out=  0.0% | MSE*1e3=  18.98 | rho=0.945
Gemini 4 Argon (High thinking, Respondent)               | n=1000 (inv=  0) | opt_out=  9.1% | MSE*1e

## 4. Score Saved Base `KEV-0.8B`, `KEV-4B`, and `KEV-9B` Predictions (Calibrated $T_{\mathrm{cal}}$ + Ablations)

In [46]:
# Precompute OLS operator once for fast 100-seed Monte Carlo ablation
sub_reg_rows = ds_covid.regression_rows(sub_choice_df)
sub_specs = ds_covid.regression_specs(sub_reg_rows)
sub_records_keyed = [{"CASEID": str(r.CASEID), "qid": str(r.qid)} for r in sub_choice_df.itertuples(index=False)]
sub_eval_ops = build_population_eval(ds_covid, sub_choice_df, sub_specs, sub_records_keyed)
B_sub = sub_eval_ops["operators"][0]["B"].numpy().astype(np.float64)

for slug, info in kev_meta["models"].items():
    display_name = info["display_name"]
    t_cal = float(info["calibrated_temperature"])
    pred_path = PREDS_DIR / info["file"]
    if not pred_path.exists():
        raise FileNotFoundError(f"Missing prediction file: {pred_path}")
    pred_df = pd.read_parquet(pred_path)

    if len(pred_df) != len(sub_choice_df):
        raise ValueError(f"{pred_path.name}: expected {len(sub_choice_df)} rows, got {len(pred_df)}")
    if not (
        (pred_df["CASEID"].astype(str).to_numpy() == sub_choice_df["CASEID"].astype(str).to_numpy()).all()
        and (pred_df["respondent"].astype(int).to_numpy() == sub_choice_df["respondent"].astype(int).to_numpy()).all()
        and (pred_df["pair_id"].astype(int).to_numpy() == sub_choice_df["pair_id"].astype(int).to_numpy()).all()
        and (pred_df["qid"].astype(str).to_numpy() == sub_choice_df["qid"].astype(str).to_numpy()).all()
    ):
        raise ValueError(f"{pred_path.name}: row keys do not match sub_choice_df order")

    p_cols = ["p_0", "p_1", "p_2"] if "p_0" in pred_df.columns else ["p_cal_0", "p_cal_1", "p_cal_2"]
    p_kev = pred_df[p_cols].to_numpy(dtype=np.float64)

    # 1. Base KEV continuous probabilities (using shipped checkpoint temperature)
    m_kev, b_kev = score_predictions_ols(sub_choice_df, list(p_kev), human_full_betas)
    summary_rows.append({
        "family": "Base decision model",
        "model": display_name,
        "config": f"Base (T={t_cal:.2f})",
        "persona": "Respondent",
        "readout": "Continuous probs",
        "refusals": 0,
        **m_kev,
    })
    betas_by_model[display_name] = b_kev

    # 2. Argmax one-hot ablation
    argmax_probs = np.zeros_like(p_kev)
    argmax_probs[np.arange(len(p_kev)), np.argmax(p_kev, axis=1)] = 1.0
    m_arg, b_arg = score_predictions_ols(sub_choice_df, list(argmax_probs), human_full_betas)
    summary_rows.append({
        "family": "Decision model ablation",
        "model": display_name,
        "config": f"Base (T={t_cal:.2f})",
        "persona": "Respondent",
        "readout": "Argmax one-hot",
        "refusals": 0,
        **m_arg,
    })
    betas_by_model[f"{display_name} (Argmax one-hot)"] = b_arg

    # 3. Single-draw Monte Carlo ablation (100-seed mean)
    p_norm = p_kev / p_kev.sum(axis=1, keepdims=True)
    cdf = np.cumsum(p_norm, axis=1)
    rng_mc = np.random.default_rng(12345)
    mc_mses = []
    for _ in range(100):
        u = rng_mc.random((len(p_norm), 1))
        draws = (u > cdf).sum(axis=1).clip(0, 2)
        oh = np.zeros_like(p_norm)
        oh[np.arange(len(p_norm)), draws] = 1.0
        mc_betas = B_sub @ oh.reshape(-1)
        mc_mses.append(float(np.mean((mc_betas - human_full_betas) ** 2) * 1000.0))
    summary_rows.append({
        "family": "Decision model ablation",
        "model": display_name,
        "config": f"Base (T={t_cal:.2f})",
        "persona": "Respondent",
        "readout": "1-draw MC (100-seed mean)",
        "refusals": 0,
        "n_tasks": 1000,
        "opt_out_pct": m_kev["opt_out_pct"],
        "beta_mse_x1e3": float(np.mean(mc_mses)),
        "beta_mae": float("nan"),
        "beta_mape_pct": float("nan"),
        "spearman_rho": float("nan"),
        "sign_acc_pct": float("nan"),
    })

    print(
        f"{display_name:9s} | MSE*1e3={m_kev['beta_mse_x1e3']:6.2f} "
        f"(opt_out={m_kev['opt_out_pct']:4.1f}%, rho={m_kev['spearman_rho']:.3f}) "
        f"| 1-draw MC={np.mean(mc_mses):6.2f} | argmax={m_arg['beta_mse_x1e3']:6.2f}"
    )

KEV-0.8B  | MSE*1e3=  5.85 (opt_out=32.1%, rho=0.500) | 1-draw MC=  6.30 | argmax= 59.23
KEV-4B    | MSE*1e3=  2.77 (opt_out=13.4%, rho=0.882) | 1-draw MC=  3.03 | argmax= 17.95
KEV-9B    | MSE*1e3=  4.38 (opt_out=16.3%, rho=0.927) | 1-draw MC=  4.66 | argmax= 28.06


## 5. Summary Table and CSV Exports (the figure is `manuscript-training-assets/fig-intro-off-the-shelf.ipynb`)

In [47]:
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUT_DIR / "results_summary.csv", index=False)

betas_df = pd.DataFrame(betas_by_model, index=term_labels)
betas_df.index.name = "term"
betas_df.to_csv(OUT_DIR / "betas_per_term.csv")

display_cols = [
    "family", "model", "config", "persona", "readout", "n_tasks", "refusals",
    "opt_out_pct", "beta_mse_x1e3", "beta_mae", "beta_mape_pct", "spearman_rho", "sign_acc_pct",
]
print("=== Respondent-Persona & Human Subsample Summary (sorted by beta_mse_x1e3) ===")
display(summary_df[summary_df["persona"] == "Respondent"][display_cols].sort_values("beta_mse_x1e3").round(3))

print("\n=== Full Summary (including Random-Persona Gemini runs) ===")
display(summary_df[display_cols].round(3))

=== Respondent-Persona & Human Subsample Summary (sorted by beta_mse_x1e3) ===


,family,model,config,persona,readout,n_tasks,refusals,opt_out_pct,beta_mse_x1e3,beta_mae,beta_mape_pct,spearman_rho,sign_acc_pct
0,Human benchmark,Survey subsample (200 resp.),Observed choices,Respondent,Observed one-hot,1000,0,20.300,0.455,0.018,34.940,0.891,100.000
12,Generative LLM (Qwen),Qwen3.5-9B,Thinking off,Respondent,Discrete text,1000,0,81.800,1.489,0.027,61.820,0.955,90.909
11,Generative LLM (Qwen),Qwen3.5-4B,Thinking off,Respondent,Discrete text,1000,0,53.600,1.937,0.029,67.294,0.955,90.909
9,Generative LLM (Qwen),Qwen3.5-0.8B,Thinking off,Respondent,Discrete text,1000,0,59.000,2.748,0.038,61.521,0.818,90.909
17,Base decision model,KEV-4B,Base (T=2.41),Respondent,Continuous probs,1000,0,13.372,2.772,0.037,68.373,0.882,100.000
19,Decision model ablation,KEV-4B,Base (T=2.41),Respondent,1-draw MC (100-seed mean),1000,0,13.372,3.031,NaN,NaN,NaN,NaN
20,Base decision model,KEV-9B,Base (T=2.19),Respondent,Continuous probs,1000,0,16.319,4.375,0.049,103.522,0.927,100.000
22,Decision model ablation,KEV-9B,Base (T=2.19),Respondent,1-draw MC (100-seed mean),1000,0,16.319,4.656,NaN,NaN,NaN,NaN
14,Base decision model,KEV-0.8B,Base (T=2.35),Respondent,Continuous probs,1000,0,32.074,5.852,0.065,114.850,0.500,72.727
13,Generative LLM (Qwen),Qwen3.5-9B,Thinking on,Respondent,Discrete text,999,1,11.411,6.266,0.066,127.088,0.973,100.000



=== Full Summary (including Random-Persona Gemini runs) ===


,family,model,config,persona,readout,n_tasks,refusals,opt_out_pct,beta_mse_x1e3,beta_mae,beta_mape_pct,spearman_rho,sign_acc_pct
0,Human benchmark,Survey subsample (200 resp.),Observed choices,Respondent,Observed one-hot,1000,0,20.300,0.455,0.018,34.940,0.891,100.000
1,Generative LLM (Gemini),Gemini 3.5 Flash-Lite,Minimal thinking,Random,Discrete text,1000,0,3.000,18.059,0.109,197.434,0.927,100.000
2,Generative LLM (Gemini),Gemini 3.5 Flash-Lite,Minimal thinking,Respondent,Discrete text,1000,0,21.900,10.642,0.083,149.699,0.936,100.000
3,Generative LLM (Gemini),Gemini 3.5 Flash-Lite,High thinking,Random,Discrete text,1000,0,0.000,19.049,0.093,156.852,0.891,100.000
4,Generative LLM (Gemini),Gemini 3.5 Flash-Lite,High thinking,Respondent,Discrete text,1000,0,4.800,12.281,0.085,151.186,0.900,100.000
5,Generative LLM (Gemini),Gemini 4 Argon,Low thinking,Random,Discrete text,1000,0,0.300,20.554,0.114,193.889,0.955,100.000
6,Generative LLM (Gemini),Gemini 4 Argon,Low thinking,Respondent,Discrete text,1000,0,12.200,16.503,0.099,169.306,0.927,100.000
7,Generative LLM (Gemini),Gemini 4 Argon,High thinking,Random,Discrete text,984,16,0.000,18.978,0.111,194.271,0.945,100.000
8,Generative LLM (Gemini),Gemini 4 Argon,High thinking,Respondent,Discrete text,1000,0,9.100,16.575,0.099,153.177,0.955,100.000
9,Generative LLM (Qwen),Qwen3.5-0.8B,Thinking off,Respondent,Discrete text,1000,0,59.000,2.748,0.038,61.521,0.818,90.909
